In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)

customers = pd.read_csv("C:/Users/lenovo/Desktop/FinRisk-X/database/customers.csv")

N_TRANSACTIONS = 200_000

transactions = pd.DataFrame({
    "transaction_id": range(900001, 900001 + N_TRANSACTIONS),

    "customer_id": np.random.choice(
        customers["customer_id"],
        N_TRANSACTIONS
    )
})

In [13]:
transactions["amount"] = np.random.lognormal(
    mean=np.log(1500),
    sigma=1.0,
    size=len(transactions)
)

transactions["amount"] = (
    transactions["amount"]
    .clip(50, 500000)
    .round(2)
)

In [3]:
start_date = pd.Timestamp("2025-01-01")
end_date = pd.Timestamp("2025-12-31")

transactions["timestamp"] = pd.to_datetime(
    np.random.randint(
        start_date.value // 10**9,
        end_date.value // 10**9,
        N_TRANSACTIONS
    ),
    unit="s"
)

transactions = transactions.sort_values(
    ["customer_id", "timestamp"]
).reset_index(drop=True)

In [4]:
start_date = pd.Timestamp("2025-01-01")
end_date = pd.Timestamp("2025-12-31")

transactions["timestamp"] = pd.to_datetime(
    np.random.randint(
        start_date.value // 10**9,
        end_date.value // 10**9,
        N_TRANSACTIONS
    ),
    unit="s"
)

transactions = transactions.sort_values(
    ["customer_id", "timestamp"]
).reset_index(drop=True)

In [5]:
transaction_types = [
    "UPI",
    "Card",
    "Bank Transfer",
    "ATM",
    "Online"
]

transactions["transaction_type"] = np.random.choice(
    transaction_types,
    N_TRANSACTIONS,
    p=[0.35, 0.30, 0.15, 0.08, 0.12]
)

In [6]:
N_MERCHANTS = 2000

transactions["merchant_id"] = np.random.randint(
    10001,
    10001 + N_MERCHANTS,
    N_TRANSACTIONS
)

In [7]:
merchant_categories = [
    "Retail",
    "Grocery",
    "Travel",
    "Food",
    "Electronics",
    "Healthcare",
    "Entertainment",
    "Financial",
    "Utilities"
]

transactions["merchant_category"] = np.random.choice(
    merchant_categories,
    N_TRANSACTIONS
)

In [8]:
N_DEVICES = 3000

transactions["device_id"] = np.random.randint(
    20001,
    20001 + N_DEVICES,
    N_TRANSACTIONS
)

In [9]:
N_DEVICES = 3000

transactions["device_id"] = np.random.randint(
    20001,
    20001 + N_DEVICES,
    N_TRANSACTIONS
)

In [10]:
transactions["is_fraud"] = 0

In [14]:
amount_threshold = transactions["amount"].quantile(0.995)

large_transaction = (
    transactions["amount"] > amount_threshold
)

In [12]:
transactions.columns.tolist()

['transaction_id',
 'customer_id',
 'timestamp',
 'transaction_type',
 'merchant_id',
 'merchant_category',
 'device_id',
 'is_fraud']

In [15]:
hour = transactions["timestamp"].dt.hour

unusual_hour = (
    (hour >= 1) &
    (hour <= 5)
)

In [16]:
fraud_signal = (
    large_transaction.astype(int)
    + unusual_hour.astype(int)
)

In [17]:
fraud_probability = np.where(
    fraud_signal >= 2,
    0.70,
    np.where(
        fraud_signal == 1,
        0.12,
        0.01
    )
)

In [18]:
transactions["is_fraud"] = (
    np.random.random(N_TRANSACTIONS)
    < fraud_probability
).astype(int)

In [19]:
transactions["is_fraud"].value_counts()

is_fraud
0    193191
1      6809
Name: count, dtype: int64

In [20]:
transactions["hour"] = transactions["timestamp"].dt.hour

In [21]:
transactions["day_of_week"] = (
    transactions["timestamp"].dt.dayofweek
)

In [22]:
transactions["is_weekend"] = (
    transactions["day_of_week"] >= 5
).astype(int)

In [23]:
transactions.to_csv(
    "C:/Users/lenovo/Desktop/FinRisk-X/database/transactions.csv",
    index=False
)